In [2]:
# ===== CONFIG =====
   # change this
MODEL_PATH = "/kaggle/input/models/purebloods/multi-cancer-unfinished/keras/default/1"
SAVE_PATH = "/kaggle/working/fine_tuned_model.keras"


In [3]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        # Must be done BEFORE any GPU usage
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        strategy = tf.distribute.MirroredStrategy()
        print(f"🔥 Using {strategy.num_replicas_in_sync} GPUs")

    except RuntimeError as e:
        print("❌ GPU init error:", e)
else:
    print("⚠️ No GPU found")
    strategy = tf.distribute.get_strategy()

❌ GPU init error: Physical devices cannot be modified after being initialized


In [4]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    strategy = tf.distribute.MirroredStrategy()
    print(f"🔥 Using {strategy.num_replicas_in_sync} GPUs")
else:
    strategy = tf.distribute.get_strategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
🔥 Using 2 GPUs


In [5]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import gc

#print("🔥 TF Version:", tf.__version__)

In [26]:
DATASET_PATH = "/kaggle/input/datasets/obulisainaren/multi-cancer/Multi Cancer/Multi Cancer"

IMG_SIZE = (224, 224)
BATCH_SIZE = 128

main_classes = sorted(os.listdir(DATASET_PATH))

image_paths = []
main_labels = []
sub_labels = []

main_class_map = {}
sub_class_map = {}

sub_index = 0

for m_idx, main_class in enumerate(main_classes):
    main_class_map[main_class] = m_idx
    main_path = os.path.join(DATASET_PATH, main_class)

    if not os.path.isdir(main_path):
        continue

    for sub_class in os.listdir(main_path):
        sub_path = os.path.join(main_path, sub_class)

        if not os.path.isdir(sub_path):
            continue

        if sub_class not in sub_class_map:
            sub_class_map[sub_class] = sub_index
            sub_index += 1

        for img in os.listdir(sub_path):
            image_paths.append(os.path.join(sub_path, img))
            main_labels.append(m_idx)
            sub_labels.append(sub_class_map[sub_class])

print("Total images:", len(image_paths))
print("Main classes:", len(main_class_map))
print("Sub classes:", len(sub_class_map))

Total images: 130002
Main classes: 8
Sub classes: 26


In [27]:
from sklearn.model_selection import train_test_split

# Convert to numpy (for splitting)
image_paths = np.array(image_paths)
main_labels = np.array(main_labels)
sub_labels = np.array(sub_labels)

# 🔹 First split: Train + Temp
X_train, X_temp, y_main_train, y_main_temp, y_sub_train, y_sub_temp = train_test_split(
    image_paths, main_labels, sub_labels,
    test_size=0.2,
    random_state=42,
    stratify=main_labels
)

# 🔹 Second split: Val + Test
X_val, X_test, y_main_val, y_main_test, y_sub_val, y_sub_test = train_test_split(
    X_temp, y_main_temp, y_sub_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_main_temp
)

print(f"Train: {len(X_train)}")
print(f"Val: {len(X_val)}")
print(f"Test: {len(X_test)}")

Train: 104001
Val: 13000
Test: 13001


In [28]:
def preprocess_image(path, main_label, sub_label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0

    return img, {
        "cancer_type": tf.one_hot(main_label, depth=len(main_class_map)),
        "subclass": tf.one_hot(sub_label, depth=len(sub_class_map))
    }


def build_dataset(paths, main_labels, sub_labels, training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, main_labels, sub_labels))

    if training:
        ds = ds.shuffle(10000)

    ds = ds.map(
        lambda x, y, z: preprocess_image(x, y, z),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_dataset = build_dataset(X_train, y_main_train, y_sub_train, training=True)
val_dataset   = build_dataset(X_val, y_main_val, y_sub_val, training=False)
test_dataset  = build_dataset(X_test, y_main_test, y_sub_test, training=False)

In [11]:
with strategy.scope():
    model = keras.models.load_model("/kaggle/working/final_fine_tuned_model.keras")

    # Freeze most layers
    for layer in model.layers[:-20]:
        layer.trainable = False

In [12]:
trainable_count = sum([int(layer.trainable) for layer in model.layers])
print("Trainable layers:", trainable_count)

Trainable layers: 4


In [32]:
with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss={
            "cancer_type": "categorical_crossentropy",
            "subclass": "categorical_crossentropy"
        },
        metrics={
            "cancer_type": "accuracy",
            "subclass": "accuracy"
        }
    )

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 113, 113,  │          0 │ activation[0][0]  │
│ (ZeroPadding2D)     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │        144 │ expanded_conv_de… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 56, 56,    │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │          0 │ re_lu[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │        136 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │          0 │ expanded_conv_sq… │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │        144 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1, 1, 16)  │          0 │ expanded_conv_sq… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 1, 1, 16)  │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1, 1, 16)  │          0 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 56, 56,    │          0 │ re_lu[0][0],      │
│ (Multiply)          │ 16)               │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 56, 56,    │        256 │ expanded_conv_sq

 Total params: 1,017,362 (3.88 MB)

 Trainable params: 78,242 (305.63 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [ ]:
EPOCHS = 15

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=1,
        verbose=1
    )
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/15
INFO:tensorflow:Collective all_reduce tensors: 6 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
813/813 ━━━━━━━━━━━━━━━━━━━━ 168s 200ms/step - cancer_type_accuracy: 0.8360 - cancer_type_loss: 0.4461 - loss: 1.7443 - subclass_accuracy: 0.5371 - subclass_loss: 1.2982 - val_cancer_type_accuracy: 0.8401 - val_cancer_type_loss: 0.4482 - val_loss: 1.7442 - val_subclass_accuracy: 0.5435 - val_subclass_loss: 1.2959 - learning_rate: 1.0000e-04
Epoch 2/15
703/813 ━━━━━━━━━━━━━━━━━━━━ 20s 183ms/step - cancer_type_accuracy: 0.8374 - cancer_type_loss: 0.4455 - loss: 1.7416 - subclass_accuracy: 0.5385 - subclass_loss: 1.2961

In [ ]:
print(model.output_names)

In [25]:
# ==============================
# EVALUATION ON TEST SET
# ==============================

print("🧪 Evaluating model on test dataset...\n")

results = model.evaluate(test_dataset, verbose=1)

print("\n📊 Evaluation Results:")
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")


# ==============================
# PREDICTIONS (FIXED)
# ==============================

import numpy as np

print("\n🔍 Running sample predictions...\n")

for batch_images, batch_labels in test_dataset.take(1):
    preds = model.predict(batch_images)

    # ✅ Access using names, not index
    cancer_pred = np.argmax(preds["cancer_type"], axis=1)
    subclass_pred = np.argmax(preds["subclass"], axis=1)

    cancer_true = np.argmax(batch_labels["cancer_type"], axis=1)
    subclass_true = np.argmax(batch_labels["subclass"], axis=1)

    for i in range(min(5, len(cancer_pred))):
        print(f"Sample {i+1}:")
        print(f"  True   → Cancer: {cancer_true[i]}, Subclass: {subclass_true[i]}")
        print(f"  Pred   → Cancer: {cancer_pred[i]}, Subclass: {subclass_pred[i]}")
        print("-" * 40)

🧪 Evaluating model on test dataset...

51/51 ━━━━━━━━━━━━━━━━━━━━ 18s 352ms/step - cancer_type_accuracy: 0.8376 - cancer_type_loss: 0.4516 - loss: 1.7584 - subclass_accuracy: 0.5334 - subclass_loss: 1.3068

📊 Evaluation Results:
loss: 1.7704
compile_metrics: 0.4603
cancer_type_loss: 1.3099
subclass_loss: 0.8315

🔍 Running sample predictions...

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Sample 1:
  True   → Cancer: 3, Subclass: 13
  Pred   → Cancer: 0, Subclass: 0
----------------------------------------
Sample 2:
  True   → Cancer: 5, Subclass: 20
  Pred   → Cancer: 5, Subclass: 20
----------------------------------------
Sample 3:
  True   → Cancer: 1, Subclass: 5
  Pred   → Cancer: 1, Subclass: 5
----------------------------------------
Sample 4:
  True   → Cancer: 3, Subclass: 12
  Pred   → Cancer: 3, Subclass: 13
----------------------------------------
Sample 5:
  True   → Cancer: 0, Subclass: 0
  Pred   → Cancer: 0, Subclass: 3
----------------------------------------


In [ ]:
SAVE_PATH = "/kaggle/working/f_model.keras"

model.save(SAVE_PATH)

print(f"\n💾 Model saved successfully at:\n{SAVE_PATH}")